In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings('ignore')

In [9]:
df = pd.read_excel('FillipsCurve.xlsx', sheet_name='Лист1')
df['Data'] = pd.to_datetime(df['Data'])
df.set_index('Data', inplace=True)
df.columns = ['core_inf', 'unemployment', 'wages', 'M1', 'exp_prices', 'imp_prices']
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [10]:
df = df[df.index >= '2009-07-01']

In [11]:
df['m_growth'] = df['M1'].pct_change() * 100
df['w_growth'] = df['wages'].pct_change() * 100
df['imp_growth'] = df['imp_prices'].pct_change() * 100

In [12]:
df = df.dropna().copy()

In [13]:
def adf_test(series, name, maxlag=6, regression='c'):
    result = adfuller(series, maxlag=maxlag, autolag=None, regression=regression)
    print(f'ADF тест для {name}:')
    print(f'  t-statistic: {result[0]:.4f}')
    print(f'  p-value: {result[1]:.4f}')
    print(f'  критическое значение 5%: {result[4]["5%"]:.4f}')
    if result[1] <= 0.05:
        print(f'  -> {name} стационарен\n')
        return True
    else:
        print(f'  -> {name} нестационарен\n')
        return False

In [14]:
inf_stat = adf_test(df['core_inf'], 'Инфляция (уровень)', maxlag=6)
unemp_stat = adf_test(df['unemployment'], 'Безработица (уровень)', maxlag=6)


ADF тест для Инфляция (уровень):
  t-statistic: -2.2778
  p-value: 0.1792
  критическое значение 5%: -2.9240
  -> Инфляция (уровень) нестационарен

ADF тест для Безработица (уровень):
  t-statistic: 1.1297
  p-value: 0.9955
  критическое значение 5%: -2.9240
  -> Безработица (уровень) нестационарен



In [15]:
if not inf_stat:
    df['core_inf_diff1'] = df['core_inf'].diff()
    inf_diff1_stat = adf_test(df['core_inf_diff1'].dropna(), 'Инфляция (1-я разность)', maxlag=6)
    if not inf_diff1_stat:
        df['core_inf_diff2'] = df['core_inf'].diff().diff()
        print("Берём вторую разность инфляции:")
        adf_test(df['core_inf_diff2'].dropna(), 'Инфляция (2-я разность)', maxlag=6)
        df['inf_stationary'] = df['core_inf_diff2']
    else:
        df['inf_stationary'] = df['core_inf_diff1']
else:
    df['inf_stationary'] = df['core_inf']

ADF тест для Инфляция (1-я разность):
  t-statistic: -2.2645
  p-value: 0.1837
  критическое значение 5%: -2.9253
  -> Инфляция (1-я разность) нестационарен

Берём вторую разность инфляции:
ADF тест для Инфляция (2-я разность):
  t-statistic: -3.9371
  p-value: 0.0018
  критическое значение 5%: -2.9268
  -> Инфляция (2-я разность) стационарен



In [16]:
if not unemp_stat:
    df['unemp_diff1'] = df['unemployment'].diff()
    print("Берём первую разность безработицы:")
    adf_test(df['unemp_diff1'].dropna(), 'Безработица (1-я разность)', maxlag=6)
    df['unemp_stationary'] = df['unemp_diff1']
else:
    df['unemp_stationary'] = df['unemployment']

Берём первую разность безработицы:
ADF тест для Безработица (1-я разность):
  t-statistic: -3.7905
  p-value: 0.0030
  критическое значение 5%: -2.9253
  -> Безработица (1-я разность) стационарен



In [17]:
df = df.dropna().copy()
print("\nСтационарность после преобразований:")
adf_test(df['inf_stationary'], 'inf_stationary', maxlag=6)
adf_test(df['unemp_stationary'], 'unemp_stationary', maxlag=6)


Стационарность после преобразований:
ADF тест для inf_stationary:
  t-statistic: -3.9371
  p-value: 0.0018
  критическое значение 5%: -2.9268
  -> inf_stationary стационарен

ADF тест для unemp_stationary:
  t-statistic: -4.4462
  p-value: 0.0002
  критическое значение 5%: -2.9268
  -> unemp_stationary стационарен



True

In [19]:
granger_data = df[['inf_stationary', 'unemp_stationary']].dropna()
maxlag = min(12, len(granger_data)//3)

if maxlag >= 1:
    print("\nПричина: безработица -> инфляция")
    gc1 = grangercausalitytests(granger_data[['inf_stationary', 'unemp_stationary']], maxlag, verbose=False)
    for lag in range(1, maxlag+1):
        pval = gc1[lag][0]['ssr_ftest'][1]
        print(f"  лаг {lag:2d}: p-value = {pval:.4f} {'*' if pval < 0.05 else ''}")

    print("\nПричина: инфляция -> безработица")
    gc2 = grangercausalitytests(granger_data[['unemp_stationary', 'inf_stationary']], maxlag, verbose=False)
    for lag in range(1, maxlag+1):
        pval = gc2[lag][0]['ssr_ftest'][1]
        print(f"  лаг {lag:2d}: p-value = {pval:.4f} {'*' if pval < 0.05 else ''}")
else:
    print("Недостаточно наблюдений для теста Грейнджера")


Причина: безработица -> инфляция
  лаг  1: p-value = 0.8538 
  лаг  2: p-value = 0.5984 
  лаг  3: p-value = 0.8031 
  лаг  4: p-value = 0.8228 
  лаг  5: p-value = 0.8090 
  лаг  6: p-value = 0.9085 
  лаг  7: p-value = 0.5991 
  лаг  8: p-value = 0.8197 
  лаг  9: p-value = 0.4499 
  лаг 10: p-value = 0.6210 
  лаг 11: p-value = 0.7562 
  лаг 12: p-value = 0.3468 

Причина: инфляция -> безработица
  лаг  1: p-value = 0.0210 *
  лаг  2: p-value = 0.0322 *
  лаг  3: p-value = 0.1238 
  лаг  4: p-value = 0.0490 *
  лаг  5: p-value = 0.2081 
  лаг  6: p-value = 0.1344 
  лаг  7: p-value = 0.1379 
  лаг  8: p-value = 0.4439 
  лаг  9: p-value = 0.5771 
  лаг 10: p-value = 0.6943 
  лаг 11: p-value = 0.6296 
  лаг 12: p-value = 0.6931 


In [20]:
inf_level = df['core_inf'].dropna()
best_aic = np.inf
best_order = 1
for p in range(1, 13):
    if len(inf_level) > p + 5:
        try:
            model = AutoReg(inf_level, lags=p, old_names=False)
            res = model.fit()
            if res.aic < best_aic:
                best_aic = res.aic
                best_order = p
        except:
            continue
print(f"Лучший порядок AR по AIC: {best_order}")
ar_model = AutoReg(inf_level, lags=best_order, old_names=False)
ar_result = ar_model.fit()
print(ar_result.summary())

Лучший порядок AR по AIC: 6
                            AutoReg Model Results                             
Dep. Variable:               core_inf   No. Observations:                   53
Model:                     AutoReg(6)   Log Likelihood                  54.356
Method:               Conditional MLE   S.D. of innovations              0.076
Date:                Wed, 08 Apr 2026   AIC                            -92.712
Time:                        12:32:43   BIC                            -77.911
Sample:                             6   HQIC                           -87.143
                                   53                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.1320      0.043      3.073      0.002       0.048       0.216
core_inf.L1     1.2570      0.131      9.611      0.000       1.001       1.513
core_inf.L2    -0.36

In [21]:
df['inf_exp'] = ar_result.fittedvalues
df['inf_exp_lag1'] = df['inf_exp'].shift(1)

In [22]:
cycle, trend = hpfilter(df['unemployment'].dropna(), lamb=1600)
df['unemp_natural'] = trend
df['unemp_gap'] = df['unemployment'] - df['unemp_natural']


In [ ]:
y = df['inf_stationary'].copy()
df['unemp_gap_diff'] = df['unemp_gap'].diff()
inf_stat_series = df['inf_stationary'].dropna()
best_aic_stat = np.inf
best_order_stat = 1
for p in range(1, 7):
    if len(inf_stat_series) > p + 5:
        try:
            model = AutoReg(inf_stat_series, lags=p, old_names=False)
            res = model.fit()
            if res.aic < best_aic_stat:
                best_aic_stat = res.aic
                best_order_stat = p
        except:
            continue
print(f"Лучший AR порядок для стационарной инфляции: {best_order_stat}")
ar_stat = AutoReg(inf_stat_series, lags=best_order_stat, old_names=False).fit()
df['inf_exp_stat'] = ar_stat.fittedvalues
df['inf_exp_stat_lag1'] = df['inf_exp_stat'].shift(1)

model_df = pd.DataFrame({
    'inf_stationary': y,
    'unemp_stationary_lag1': df['unemp_stationary'].shift(1),
    'm_growth_lag1': df['m_growth'].shift(1),
    'w_growth_lag1': df['w_growth'].shift(1),
    'imp_growth_lag1': df['imp_growth'].shift(1),
    'inf_exp_stat_lag1': df['inf_exp_stat_lag1']
}).dropna()

X = model_df[['unemp_stationary_lag1', 'm_growth_lag1', 'w_growth_lag1', 'imp_growth_lag1', 'inf_exp_stat_lag1']]
X = sm.add_constant(X)
y = model_df['inf_stationary']

model_phillips = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(model_phillips.summary())

residuals = model_phillips.resid
adf_res = adfuller(residuals, maxlag=6, autolag=None, regression='c')
print(f"\nADF тест остатков модели: p-value = {adf_res[1]:.4f}")
if adf_res[1] < 0.05:
    print("Остатки стационарны → модель корректна (коинтеграция)")
else:
    print("Остатки нестационарны → возможна ложная регрессия (но все переменные I(0), поэтому это менее вероятно)")

# Тест Льюнга-Бокса на автокорреляцию
lb_test = acorr_ljungbox(residuals, lags=[4, 8, 12], return_df=True)
print("\nТест Льюнга-Бокса для остатков (p-value):")
print(lb_test)

Лучший AR порядок для стационарной инфляции: 4
                            OLS Regression Results                            
Dep. Variable:         inf_stationary   R-squared:                       0.111
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     4.417
Date:                Wed, 08 Apr 2026   Prob (F-statistic):            0.00254
Time:                        12:34:25   Log-Likelihood:                 42.429
No. Observations:                  48   AIC:                            -72.86
Df Residuals:                      42   BIC:                            -61.63
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------

In [26]:
print(f"""
R-squared: {model_phillips.rsquared:.4f}
Adj. R-squared: {model_phillips.rsquared_adj:.4f}
F-статистика: {model_phillips.fvalue:.2f} (p = {model_phillips.f_pvalue:.4f})
Число наблюдений: {int(model_phillips.nobs)}

Коэффициенты (все переменные стационарны):
- Константа: {model_phillips.params[0]:.4f} (p={model_phillips.pvalues[0]:.4f})
- Δ unemployment (t-1): {model_phillips.params[1]:.4f} (p={model_phillips.pvalues[1]:.4f})
- Δ M1 (t-1): {model_phillips.params[2]:.4f} (p={model_phillips.pvalues[2]:.4f})
- Δ wages (t-1): {model_phillips.params[3]:.4f} (p={model_phillips.pvalues[3]:.4f})
- Δ import prices (t-1): {model_phillips.params[4]:.4f} (p={model_phillips.pvalues[4]:.4f})
- Ожидаемая инфляция (t-1, стационарная): {model_phillips.params[5]:.4f} (p={model_phillips.pvalues[5]:.4f})

Выводы:
1. Ожидаемая инфляция {'сильно значима' if model_phillips.pvalues[5] < 0.01 else 'незначима'}.
2. Изменение безработицы {'значимо' if model_phillips.pvalues[1] < 0.05 else 'незначимо'} влияет на изменение инфляции.
3. Прирост денежной массы {'значим' if model_phillips.pvalues[2] < 0.05 else 'незначим'}.
4. Прирост зарплат {'значим' if model_phillips.pvalues[3] < 0.05 else 'незначим'}.
5. Прирост импортных цен {'значим' if model_phillips.pvalues[4] < 0.05 else 'незначим'}.
6. Модель объясняет {model_phillips.rsquared*100:.1f}% дисперсии изменения инфляции.
7. Остатки {'стационарны' if adf_res[1] < 0.05 else 'нестационарны'}, что {'подтверждает' if adf_res[1] < 0.05 else 'ставит под сомнение'} корректность спецификации.
""")


R-squared: 0.1114
Adj. R-squared: 0.0056
F-статистика: 4.42 (p = 0.0025)
Число наблюдений: 48

Коэффициенты (все переменные стационарны):
- Константа: 0.0171 (p=0.3327)
- Δ unemployment (t-1): 0.0486 (p=0.6163)
- Δ M1 (t-1): -0.0180 (p=0.1883)
- Δ wages (t-1): 0.0801 (p=0.3558)
- Δ import prices (t-1): 0.0341 (p=0.0010)
- Ожидаемая инфляция (t-1, стационарная): 0.0934 (p=0.7000)

Выводы:
1. Ожидаемая инфляция незначима.
2. Изменение безработицы незначимо влияет на изменение инфляции.
3. Прирост денежной массы незначим.
4. Прирост зарплат незначим.
5. Прирост импортных цен значим.
6. Модель объясняет 11.1% дисперсии изменения инфляции.
7. Остатки стационарны, что подтверждает корректность спецификации.

